# Resultados dos experimentos

Execute **Run All** para visualizar o resumo publicado e os artefatos disponíveis no checkout. O painel não executa inferência nem treinamento.

O resumo é um snapshot histórico; seções detalhadas aparecem quando os registros de execução são fornecidos. Ausência de arquivos não informa o estado atual de jobs.

## Como ler os acertos e erros

**Acurácia = respostas aceitas pelo corretor ÷ total de perguntas.** A tabela ilustra a métrica principal do piloto. Um zero por formato pode conter o fato correto. A revisão semântica verifica esse fato na resposta bruta.

| Gabarito e regra | Saída do modelo | Nota | Motivo |
|---|---|---:|---|
| 85, intervalo inclusivo de 80,75 a 89,25 | `82` | 1 | Valor dentro do intervalo. |
| Mesmo gabarito | `90` | 0 | Valor fora do intervalo. |
| Mesmo gabarito | `A barra vale 85.` | 0 | O valor está correto, mas o corretor exige uma resposta numérica isolada. |
| Sim, pergunta binária | `Yes` | 1 | A regra principal aceita o equivalente em inglês. |
| Grupo F, categoria | `O maior é o Grupo F.` | 0 | A categoria está correta, mas a frase inteira difere do rótulo esperado. |
| A, B, C, em ordem pedida | `B, A, C` | 0 | A resposta inverte duas posições. |

Nos pares A/B, apenas dois acertos contam como par correto. Nos resultados históricos, confira a versão do corretor exibida para cada medida.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, html, hashlib
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown, Image

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'configs').is_dir() and (p / 'src/evaluation').is_dir())
RESULTS = ROOT / 'results'
INCLUIR_SMOKE = False  # True: mostrar também testes técnicos
MODELO = ''           # filtro opcional: llava, qwen, mof13b
plt.rcParams.update({'figure.figsize': (10, 4), 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 11})

def read(path):
    return json.loads(path.read_text(encoding='utf-8'))

def table(rows):
    if not rows:
        display(Markdown('Ainda não há artefatos locais para esta seção.'))
        return
    keys = list(rows[0])
    esc = lambda x: html.escape(str(x))
    display(HTML('<div style="overflow-x:auto"><table><thead><tr>' +
                 ''.join('<th>'+esc(k)+'</th>' for k in keys) + '</tr></thead><tbody>' +
                 ''.join('<tr>'+''.join('<td>'+esc(r.get(k, '—'))+'</td>' for k in keys)+'</tr>'
                         for r in rows) + '</tbody></table></div>'))

def pct(x):
    return f'{100*x:.2f}%' if isinstance(x, (float, int)) else '—'

folders = sorted(p for p in RESULTS.iterdir() if p.is_dir() and (p/'run.json').exists())
runs = []
issues = []
for folder in folders:
    try:
        run = read(folder/'run.json')
        if run.get('training_kind') == 'full_sft':
            continue  # Esta série tem seção própria abaixo.
        if (run.get('smoke', False) or 'smoke' in folder.name) and not INCLUIR_SMOKE:
            continue
        if MODELO and MODELO not in folder.name:
            continue
        complete = read(folder/'complete.json') if (folder/'complete.json').exists() else None
        pred = folder/'predictions.jsonl'
        count = sum(bool(line.strip()) for line in pred.open()) if pred.exists() else 0
        artifacts = [folder/name for name in ('run.json','predictions.jsonl','complete.json','scored/metrics.json')]
        updated_at = max((p.stat().st_mtime for p in artifacts if p.exists()), default=folder.stat().st_mtime)
        integrity = 'não verificada / parcial'
        if complete and complete.get('predictions_sha256'):
            valid = pred.exists() and hashlib.sha256(pred.read_bytes()).hexdigest() == complete['predictions_sha256']
            integrity = 'hash confere' if valid else 'DIVERGÊNCIA'
            if not valid: issues.append(folder.name + ': hash das previsões diverge')
        runs.append({'folder':folder, 'run':run, 'complete':complete, 'count':count, 'integrity':integrity,'updated_at':updated_at})
    except (ValueError, OSError) as exc:
        issues.append(folder.name + ': ' + str(exc))
display(Markdown('**Leitura local em UTC:** ' + datetime.now(timezone.utc).isoformat(timespec='seconds') +
                 '\n\n**Raiz:** `' + str(ROOT) + '`'))


## Resumo publicado
Métricas por pergunta e por par são apresentadas em tabelas distintas. As configurações sem resultado final continuam sem pontuação.

In [ ]:
published_path = RESULTS / 'published/summary.json'
if published_path.exists():
    published = read(published_path)
    display(Markdown('**Snapshot de evidências:** ' + published['evidence_snapshot_date'] +
                     '. Não representa consulta ao scheduler.'))
    table([{'Modelo':r['model'], 'Configuração':r['configuration'],
            'Antes':pct(r['before']['accuracy']) if r['before'] else 'não confirmado',
            'Depois':pct(r['after']['accuracy']) if r['after'] else 'sem resultado publicado',
            'Acertos após':str(r['after']['correct'])+'/'+str(r['after']['n']) if r['after'] else '—'}
           for r in published['pilot_results']])
    baseline_public = published['official_baseline']
    table([{'I-MoF 13B':rule, 'N':v['n'], 'Acertos':v['correct'], 'Acurácia':pct(v['accuracy'])}
           for rule,v in [('corretor original',baseline_public['strict_v1']),
                          ('diagnóstico posterior de idioma',baseline_public['binary_aliases_posthoc'])]])
    table([{'Modelo':r['model'], 'Condição':r['condition'],
            'Por pergunta':pct(r['question_accuracy']), 'Por par':pct(r['pair_accuracy']),
            'IC95% por par':str([round(100*x,2) for x in r['pair_ci95']]),
            'Inválidas':r['invalid_responses']} for r in published['paired_results']])
else:
    display(Markdown('Resumo publicado indisponível. Consulte results/published/README.md.'))


## Método e limites

Acurácia por pergunta = acertos / perguntas; acurácia por par exige os dois acertos. A métrica legada utiliza comparação normalizada e tolerância numérica do gabarito; **não é BERTScore nem revisão semântica**. Não comparar diretamente tarefas, corretores ou amostras diferentes. Uma futura revisão do corretor deve gerar outra versão, preservando o resultado original.

### Premissas
“Completo” abaixo significa presença do marcador local; o hash das previsões é conferido quando registrado. Isso não certifica validade científica. Ausência local não significa falha no cluster. Os horários são de arquivos locais, não o estado atual da GPU.

## Inventário e cobertura local


In [ ]:
table([{'Execução': x['folder'].name,
        'Escopo': 'teste técnico' if x['run'].get('smoke') or 'smoke' in x['folder'].name else 'experimento',
        'Estado local': 'marcador completo' if x['complete'] else 'parcial / sem marcador',
        'Respostas salvas': x['count'], 'Integridade': x['integrity'],
        'Atualizado (UTC)': datetime.fromtimestamp(x['updated_at'], timezone.utc).strftime('%Y-%m-%d %H:%M')}
       for x in runs])
for issue in issues:
    display(Markdown('**Atenção:** ' + issue))


## Baseline I-MoF: acurácia e erros por tipo de gráfico

In [ ]:
baseline = RESULTS / 'mof13b-pilot50-v1/scored/metrics.json'
if baseline.exists():
    b = read(baseline)
    table([{'Métrica': 'Acurácia legada por pergunta', 'Valor': pct(b['accuracy']),
            'Acertos / total': f"{b['correct']} / {b['n']}",
            'Ausentes ou falhas': b['missing_or_failed']}])
    groups = b['by_tipo']
    fig, ax = plt.subplots()
    bars = ax.bar(list(groups), [100*v['accuracy'] for v in groups.values()], color='#3676ab')
    ax.bar_label(bars, labels=[f"{v['correct']}/{v['n']}" for v in groups.values()], padding=4)
    ax.set(ylabel='Acurácia (%)', title='I-MoF 13B — corretor legado; 360 gráficos / 3.173 perguntas', ylim=(0, 100))
    fig.tight_layout(); plt.show()
    table([{'Tipo':k, 'Perguntas':v['n'], 'Acertos':v['correct'], 'Acurácia':pct(v['accuracy'])}
           for k,v in groups.items()])
else:
    display(Markdown('Baseline ainda não sincronizado.'))

audit_path = RESULTS/'mof13b-pilot50-v1/diagnostics/numeric_format_audit_v1.json'
if audit_path.exists():
    audit = read(audit_path)
    display(Markdown('### Auditoria de formato — coorte de erros do baseline original'))
    labels = {'four_values_between_zero_and_one':'Quatro valores entre 0 e 1 em pergunta de IQR',
              'interval_instead_of_frequency':'Faixa no lugar de frequência',
              'bar_description_with_incompatible_values':'Descrição com valores incompatíveis'}
    table([{'Tipo de resposta':labels.get(k,k), 'Casos':v} for k,v in audit['shape_counts'].items()])
    display(Markdown(f"Alvos conferidos nos CSVs: **{audit['csv_targets_verified']}**. "
                     f"Respostas com número literal no intervalo aceito: **{audit['responses_with_explicit_literal_in_target_interval']}/{audit['cohort_size']}**. "
                     'Coorte selecionada por erro; não é uma nova métrica semântica nem revisão de todas as respostas. '
                     'Detalhes em `results/mof13b-pilot50-v1/NUMERIC_FORMAT_AUDIT.md`.'))


## Controles antes e depois do treinamento
Os braços só são exibidos quando há métricas locais. A comparação final deve usar a mesma amostra e o mesmo corretor; não interpretar adaptadores sem treino como equivalentes ao I-MoF treinado.

Somente previsões completas com hash conferido entram nas tabelas e figuras desta seção.
Testes técnicos, quando habilitados, aparecem em figuras próprias.


In [ ]:
control_rows = []
control_plot = []
for item in runs:
    path = item['folder']/'scored/metrics.json'
    if not path.exists() or item['integrity'] != 'hash confere': continue
    m = read(path)
    for arm, values in m.get('metrics', {}).items():
        if not isinstance(values, dict): continue
        score = values.get('primary_binary_aliases_v2')
        if score:
            control_rows.append({'Execução':item['folder'].name, 'Braço':arm,
                                 'Corretor':m.get('scorer_version','—'),
                                 'Perguntas':score['n'], 'Acurácia':pct(score['accuracy'])})
            control_plot.append({'model':item['run']['model'], 'arm':arm,
                                 'scope':'teste técnico' if item['run'].get('smoke') or 'smoke' in item['folder'].name else 'piloto',
                                 'trained':False, 'score':score})
for item in runs:
    path = item['folder']/'scored/metrics.json'
    if not path.exists() or item['integrity'] != 'hash confere': continue
    metric = read(path)
    score = metric.get('primary_binary_aliases_v2')
    if score:
        arm = item['run'].get('arm','treinado')
        control_rows.append({'Execução':item['folder'].name, 'Braço':arm,
                             'Corretor':'binary_aliases_v2', 'Perguntas':score['n'],
                             'Acurácia':pct(score['accuracy'])})
        control_plot.append({'model':item['run']['model'], 'arm':arm,
                                 'scope':'teste técnico' if item['run'].get('smoke') or 'smoke' in item['folder'].name else 'piloto',
                             'trained':True, 'score':score})
table(control_rows)
arm_labels = {'native':'Nativo', 'native_double':'Nativo duplicado',
              'clip_double':'CLIP+CLIP', 'dino_double':'DINOv3+DINOv3',
              'clip_dino':'CLIP+DINOv3', 'clip_dino_shuffled':'CLIP+DINOv3 embaralhado'}
for model,scope in sorted({(row['model'],row['scope']) for row in control_plot}):
    points = [row for row in control_plot if row['model'] == model and row['scope'] == scope]
    names = [arm_labels.get(row['arm'], row['arm']) + (' — ajustado' if row['trained'] else '')
             for row in points]
    values = [100*row['score']['accuracy'] for row in points]
    fig, ax = plt.subplots(figsize=(10,max(3,0.5*len(points))), layout='constrained')
    ax.barh(range(len(points)), values, color=['#27765b' if row['trained'] else '#3579ad' for row in points])
    ax.set(yticks=range(len(points)), yticklabels=names, xlim=(0,100),
           xlabel='Acurácia por pergunta (%)',
           title=model.upper()+' — '+scope+'; controles e checkpoints completos disponíveis')
    ax.invert_yaxis()
    for i,(value,row) in enumerate(zip(values,points)):
        score=row['score']
        ax.text(value-1 if value>85 else value+1,i,
                f"{value:.2f}% ({score['correct']}/{score['n']})",va='center',
                ha='right' if value>85 else 'left',color='white' if value>85 else '#222')
    ax.spines[['top','right']].set_visible(False)
    plt.show()
comparison = RESULTS/'alignment-comparison-v1'
if comparison.exists():
    display(Markdown('Comparação consolidada disponível em `' + str(comparison.relative_to(ROOT)) + '`.'))


## Estrutura das respostas em lista

Diagnóstico posterior de 120 perguntas por execução/braço, preservando o corretor oficial. Uma permutação completa contém cada rótulo exatamente uma vez; a ordem ainda precisa estar correta. Fragmentos não reconhecidos podem ser texto, formatação ou rótulos extras. Contagens podem se sobrepor e não substituem revisão semântica.

In [ ]:
list_path = RESULTS/'list-structure-audit-v1/list_structure.json'
if list_path.exists():
    list_audit = read(list_path)
    sources_ok = all((ROOT/name).is_file() and hashlib.sha256((ROOT/name).read_bytes()).hexdigest() == expected
                     for name, expected in list_audit['sources'].items())
    if not sources_ok:
        display(Markdown('**Diagnóstico de listas desatualizado: uma fonte divergiu.**'))
    else:
        table([{'Execução / braço':key, 'N':v['n'], 'Exatas':v['exact_correct'],
                'Completas, ordem errada':v['complete_permutation_wrong_order'],
                'Fragmentos não reconhecidos':v['unrecognized_fragments'],
                'Rótulos ausentes':v['missing_target_labels'],
                'Rótulos repetidos':v['duplicate_recognized_labels']}
               for key,v in list_audit['groups'].items() if not MODELO or MODELO in key])
        display(Markdown('Detalhes e limites: `results/list-structure-audit-v1/REPORT.md`. '
                         'Análise de texto pelo parser congelado; não inclui revisão visual ou de CSV nesta etapa.'))
else:
    display(Markdown('Diagnóstico de listas ainda não sincronizado.'))

## Pares A/B e mitigações

In [ ]:
paired_rows = []
paired_plot = []
for item in runs:
    path = item['folder']/'scored/metrics.json'
    if not path.exists(): continue
    m = read(path)
    for condition, values in m.get('metrics', {}).items():
        if not isinstance(values, dict) or 'paired_accuracy' not in values: continue
        pair = values['paired_accuracy']
        ci = pair.get('ci95')
        paired_rows.append({'Execução':item['folder'].name, 'Condição':condition,
                            'Pares':pair.get('pairs','—'), 'Acurácia por pergunta':pct(values.get('question_accuracy')),
                            'Acurácia por par':pct(pair.get('estimate')),
                            'IC95% por família': ' – '.join(map(pct,ci)) if ci else '—',
                            'Inválidas':values.get('invalid_responses','—')})
        paired_plot.append((item['folder'].name.replace('paired-mitigations-',''),condition,pair))
table(paired_rows)
if paired_plot:
    fig, ax = plt.subplots(figsize=(10, max(3, len(paired_plot)*0.45)))
    for i,(run,condition,pair) in enumerate(paired_plot):
        value=pair['estimate']; ci=pair.get('ci95')
        ax.barh(i,100*value,color='#3676ab')
        if ci: ax.errorbar(100*value,i,xerr=[[100*(value-ci[0])],[100*(ci[1]-value)]],fmt='none',ecolor='black',capsize=3)
    ax.set_yticks(range(len(paired_plot)),[r+' / '+c for r,c,_ in paired_plot])
    ax.set(xlim=(0,100),xlabel='Pares com ambas as respostas corretas (%)',title='Pares A/B — IC95% por família')
    fig.tight_layout(); plt.show()


## Hipóteses H2–H3
Diferenças em pontos percentuais; IC95% por família. H2: ampla menos sutil. H3: mitigação menos original. Resultados exploratórios.

In [ ]:
hypothesis_rows = []
for model in ['llava','qwen']:
    path = RESULTS / ('paired-mitigations-full-v1-'+model) / 'scored/hypotheses.json'
    if not path.exists(): continue
    hypothesis = read(path)
    for condition, value in hypothesis['H2_and_similarity_error'].items():
        effect = value['H2']['all']
        hypothesis_rows.append({'Modelo':model,'Hipótese':'H2','Condição':condition,
                                'Diferença (pp)':round(100*effect['estimate'],2),
                                'IC95% (pp)':str([round(100*v,2) for v in effect['ci95']])})
    for condition,value in hypothesis['H3'].items():
        effect = value['paired_delta']
        hypothesis_rows.append({'Modelo':model,'Hipótese':'H3','Condição':condition,
                                'Diferença (pp)':round(100*effect['estimate'],2),
                                'IC95% (pp)':str([round(100*v,2) for v in effect['ci95']])})
table(hypothesis_rows)
display(Markdown('Relatório completo: `results/paired-comparison-v1/REPORT.md`. Correlações indefinidas não são zeros.'))

## Representações CLIP × DINOv3

In [ ]:
representation = RESULTS/'representations-pairs_v1-v1'
report = representation/'similarity_report.json'
if report.exists():
    similarity = read(report)
    table([{'Pares de avaliação':similarity.get('pairs'), 'Todos preservados':similarity.get('all_pairs_retained')}])
    table([{'Quantil de calibração':q, **values['counts']} for q,values in similarity.get('sensitivity',{}).items()])
    image = representation/'plots/similarity.png'
    if image.exists(): display(Image(filename=str(image), width=1000))
else:
    display(Markdown('Representações ainda não sincronizadas.'))


## Treinamento: perdas, validação e checkpoints

### Atualizações durante a época
Logs locais sincronizados; não representam consulta ao SLURM. A perda é a média acumulada de treino registrada naquele ponto, não perda de validação. Pontos esparsos não permitem concluir estabilidade entre registros.

In [ ]:
update_logs = sorted((ROOT/'logs').glob('sih-alignment-train_*.out'))
update_series=[]
for path in update_logs:
    points={}
    for line in path.read_text(errors='replace').splitlines():
        if not line.startswith('{'): continue
        try:
            row=json.loads(line)
        except ValueError:
            continue
        if row.get('phase')=='train' and all(k in row for k in ['updates','epoch','cursor','loss']):
            points[row['updates']]=row
    if points:
        update_series.append((path,sorted(points.values(),key=lambda v:v['updates'])))
table([{'Log':str(path.relative_to(ROOT)),'Última época':rows[-1]['epoch'],
        'Itens processados na época':rows[-1]['cursor'],'Atualizações':rows[-1]['updates'],
        'Perda média acumulada':round(rows[-1]['loss'],4),
        'Norma do gradiente':rows[-1].get('gradient_norm','—'),
        'Snapshot local (UTC)':datetime.fromtimestamp(path.stat().st_mtime,timezone.utc).isoformat(timespec='seconds')}
       for path,rows in update_series])
if update_series:
    fig,ax=plt.subplots()
    for path,rows in update_series:
        ax.plot([row['updates'] for row in rows],[row['loss'] for row in rows],marker='o',label=path.stem)
    ax.set(xlabel='Atualizações do otimizador',ylabel='Perda média acumulada na época',
           title='Treinamento — registros disponíveis nos logs locais')
    ax.legend();fig.tight_layout();plt.show()


In [ ]:
initial_rows=[]

training_inventory=[]
for run_path in sorted((ROOT/'experiments/training').glob('*/run.json')):
    folder=run_path.parent
    history=read(folder/'history.json') if (folder/'history.json').exists() else []
    completed_epochs=[row['epoch'] for row in history if row.get('phase')=='epoch_complete']
    status=('marcador local presente' if (folder/'complete.json').exists()
            else 'histórico parcial' if history else 'registro sem validação publicada')
    training_inventory.append({'Treino':folder.name, 'Situação dos arquivos locais':status,
                               'Épocas validadas':str(completed_epochs),
                               'Atualização UTC':datetime.fromtimestamp(max(p.stat().st_mtime for p in
                                   [run_path, folder/'history.json', folder/'complete.json'] if p.exists()),
                                   timezone.utc).isoformat(timespec='seconds')})
table(training_inventory)
display(Markdown('Inventário de arquivos sincronizados; não consulta jobs ativos. '
                 'Sem histórico de validação significa resultado ainda não disponível localmente. '
                 'O marcador local de término não substitui a auditoria do checkpoint.'))

for path in sorted((ROOT/'experiments/training').glob('*/validation/epoch-00/metrics.json')):
    metric=read(path)
    initial_rows.append({'Treino':path.parents[2].name,'Etapa':'validação antes do treino',
                         'Perguntas':metric['questions'],'Acurácia':pct(metric['accuracy']),
                         'Loss':round(metric['loss'],4)})
table(initial_rows)
histories = sorted((ROOT/'experiments').rglob('history.json')) + sorted(RESULTS.rglob('history.json'))
histories = [p for p in histories if 'full_sft' not in p.parts]
if not histories:
    display(Markdown('Sem históricos de treino locais. Sincronize os artefatos do Apuana para acompanhar as curvas.'))
for path in histories:
    history = read(path)
    display(Markdown('### `' + str(path.relative_to(ROOT)) + '`'))
    entries = history if isinstance(history,list) else history.get('history',[])
    entries = [dict(row, **{'dev_loss':row['development']['loss'],
                            'dev_accuracy':row['development']['accuracy']})
               if isinstance(row.get('development'),dict) else row for row in entries]
    table([{k: (json.dumps(v,ensure_ascii=False) if isinstance(v,(dict,list)) else v)
            for k,v in row.items()} for row in entries[-6:]])
    numeric_keys = sorted({k for row in entries for k,v in row.items()
                           if isinstance(v,(float,int)) and ('loss' in k or 'accuracy' in k)})
    if numeric_keys:
        fig, axes = plt.subplots(1,len(numeric_keys),figsize=(5*len(numeric_keys),3),squeeze=False)
        for ax,key in zip(axes[0],numeric_keys):
            points=[(row.get('epoch',i),row[key]) for i,row in enumerate(entries) if isinstance(row.get(key),(int,float))]
            ax.plot([p[0] for p in points],[p[1] for p in points],marker='o');ax.set(xlabel='Época',title=key)
        fig.tight_layout();plt.show()


## Relatórios, custos e auditoria
Os arquivos abaixo preservam detalhes que não devem ser reduzidos a uma única nota: latência, consumo, testes de hipóteses, revisão de leitura e comparação de treinamento.


In [ ]:
links=[]
for pattern in ['*/REPORT.md','*/DIAGNOSTICS.md','*/NUMERIC_FORMAT_AUDIT.md','*/MATCHED_COMPARISON.json','*/technical_diagnosis.json','*/run.json','*/runtime.json','*/review/index.html','*/hypotheses.json']:
    for path in sorted(RESULTS.glob(pattern)):
        if not INCLUIR_SMOKE and 'smoke' in str(path): continue
        links.append({'Artefato':str(path.relative_to(ROOT)), 'Tamanho (KB)':round(path.stat().st_size/1024,1)})
table(links)
display(Markdown('**Registro da campanha:** `experiments/five-stage-campaign/PROGRESS.md` · '
                 '**Revisão de leitura:** `experiments/five-stage-campaign/READING_REVIEW.md`'))

# Custos registrados; não estimar custos ausentes.
cost_rows=[]
for item in runs:
    path=item['folder']/'scored/metrics.json'
    if path.exists():
        metric=read(path)
        if isinstance(metric.get('cost'),dict):
            cost_rows.append({'Execução':item['folder'].name, **metric['cost']})
table(cost_rows)


## Cobertura da revisão visual
Auditoria de proveniência e cobertura; não certifica automaticamente os julgamentos visuais. Categorias podem se sobrepor. Não extrapolar frequências para o dataset.

In [ ]:
audit_path=ROOT/'experiments/five-stage-campaign/reading_review_audit_current_v1.json'
if audit_path.exists():
    audit=read(audit_path)
    table([{'Modelo':model,'Imagens revisadas':v['reviewed_images'],
            'Amostra fixa':v['sample_images'],'Restantes':v['remaining_images'],
            'Revisão completa':v['complete'],'Por tipo':str(v['by_type'])}
           for model,v in audit['models'].items()])
else:
    display(Markdown('Auditoria de revisão ainda não disponível.'))
summary_path=ROOT/'experiments/five-stage-campaign/reading_review_summary_v1.json'
if summary_path.exists():
    summary=read(summary_path)
    table([{'Modelo':model,'Imagens revisadas':v['reviewed_images'],**v['transition_counts']}
           for model,v in summary['models'].items()])
    display(Markdown('Transições de acerto por imagem, somente entre os casos revisados. '
                     'Consulte a cobertura acima; não extrapolar estas contagens para prevalência no dataset. '
                     'Síntese: `experiments/five-stage-campaign/READING_REVIEW_SUMMARY.md`.'))


## Como manter atualizado

1. Sincronize os resultados e registros do Apuana para `results/` e `experiments/`.
2. Execute **Run All** neste notebook com o kernel `.venv`.
3. Ative `INCLUIR_SMOKE` apenas para inspecionar testes técnicos.

O painel não modifica previsões, checkpoints ou métricas. Resultados pendentes não recebem nota zero. A versão atual do corretor deve ser auditada antes de atribuir o baixo resultado apenas ao modelo.


## Resultados publicados do treino em duas etapas
As linhas incluem o conjunto de avaliação e a semente. O acerto segue o corretor por formato descrito no início.

In [ ]:
public_sft_rows = []
public_sft_differences = []
for path in sorted((RESULTS/'published/full_sft').glob('*.json')):
    report = read(path)
    if report.get('schema') != 'full_sft_aggregates_v1':
        continue
    if MODELO and MODELO not in report['model']:
        continue
    group = {'Modelo': report['model'], 'Conjunto': report['dataset'], 'Semente': report['seed']}
    for arm, values in report['arms'].items():
        public_sft_rows.append({**group, 'Configuração': arm,
                               'Acertos / perguntas': f"{values['correct']} / {values['questions']}",
                               'Acurácia automática': pct(values['accuracy']),
                               'IC 95% por gráfico': ' a '.join(pct(x) for x in values['accuracy_ci95_graph_bootstrap'])})
    for control, values in report['fusion_minus_control'].items():
        interval = values['ci95_pp']
        public_sft_differences.append({**group, 'Controle': control, 'Ganho da fusão (pp)': round(values['difference_pp'],2),
                                       'IC 95% por família (pp)': ' a '.join(f'{x:.2f}' for x in interval) if interval else '—'})
if public_sft_rows:
    table(public_sft_rows)
    table(public_sft_differences)
else:
    display(Markdown('Aguardando a publicação dos resultados dos três controles.'))


## Adaptação em duas etapas com decoder treinável
Série separada da ablação de projetores. A etapa 1 usa descrições dos CSVs de treino; a etapa 2 ajusta decoder, head e projetores. Os arquivos completos permanecem no Apuana. As tabelas abaixo mostram somente artefatos acessíveis nesta cópia; ausência de arquivos não indica estado do scheduler.

In [ ]:
full_inventory=[]
for run_path in sorted((ROOT/'experiments/full_sft').glob('*/run.json')):
    folder=run_path.parent;run=read(run_path)
    if MODELO and MODELO not in run.get('model',''):continue
    progress=read(folder/'progress.json') if (folder/'progress.json').exists() else {}
    best=read(folder/'best.json') if (folder/'best.json').exists() else {}
    full_inventory.append({'Execução':folder.name,'Modelo':run['model'],'Braço':run['arm'],
        'Fase registrada':progress.get('phase','—'),'Etapa':progress.get('stage','—'),
        'Época':progress.get('epoch','—'),'Atualizações':progress.get('updates','—'),
        'Época SFT selecionada':best.get('epoch','—'),'Marcador final':(folder/'complete.json').exists()})
    if (folder/'history.json').exists():
        history=read(folder/'history.json')
        display(Markdown('### '+folder.name))
        fig,axes=plt.subplots(1,3,figsize=(15,3))
        for stage,ax in [(1,axes[0]),(2,axes[1])]:
            entries=[r for r in history if r.get('stage')==stage and r.get('phase')=='epoch_complete']
            ax.plot([r['epoch'] for r in entries],[r['train_loss'] for r in entries],marker='o')
            ax.set(title='Alinhamento: loss de descrição' if stage==1 else 'SFT: loss de QA',xlabel='Época')
        validated=[r for r in history if isinstance(r.get('development'),dict)]
        axes[2].plot([r['epoch'] for r in validated],[100*r['development']['accuracy'] for r in validated],marker='o')
        axes[2].set(title='SFT: acurácia de desenvolvimento (%)',xlabel='Época')
        fig.tight_layout();plt.show()
table(full_inventory)
full_results=[]
for run_path in sorted(RESULTS.glob('*/run.json')):
    run=read(run_path)
    if run.get('training_kind')!='full_sft':continue
    if MODELO and MODELO not in run.get('model',''):continue
    folder=run_path.parent
    if not (folder/'complete.json').exists() or not (folder/'metrics.json').exists():continue
    metrics=read(folder/'metrics.json')['primary_binary_aliases_v2']
    full_results.append({'Execução':folder.name,'Modelo':run['model'],'Braço':run['arm'],
        'Conjunto':run.get('dataset','pilot50_v1'),'Perguntas':metrics['n'],'Acurácia por pergunta':pct(metrics['accuracy']),
        'Época selecionada':run['selected_epoch']})
table(full_results)


### Contribuição do DINOv3 após o treino completo
A diferença é a acurácia da fusão menos a do controle, nas mesmas perguntas. O intervalo de 95% reamostra famílias inteiras de gráficos.

In [ ]:
comparisons = []
for path in sorted(RESULTS.glob('*-comparison/*.json')):
    report = read(path)
    if not isinstance(report, dict) or 'comparisons' not in report or 'arms' not in report:
        continue
    if MODELO and MODELO not in report['model']:
        continue
    for control, comparison in report['comparisons'].items():
        effect = comparison['fusion_minus_control_by_family']
        interval = effect['ci95']
        comparisons.append({'Campanha': report['prefix'], 'Modelo': report['model'],
                            'Conjunto': report.get('dataset', 'pilot50_v1'), 'Controle': control,
                            'Ganho da fusão (pp)': round(effect['percentage_points'], 2),
                            'IC 95% (pp)': ' a '.join(f'{100*x:.2f}' for x in interval) if interval else '—',
                            'Perguntas': effect['questions']})
if comparisons:
    table(comparisons)
else:
    display(Markdown('Aguardando os três pilotos completos de uma mesma base.'))


### Estabilidade entre sementes
Cada linha resume a diferença entre fusão e controle nos mesmos gráficos, após treinar três vezes. O desvio-padrão mede a variação entre as três sementes.

In [ ]:
seed_rows = []
for path in sorted(RESULTS.glob('*-stability/*.json')):
    report = read(path)
    if MODELO and MODELO not in report['model']:
        continue
    for arm, effect in report['fusion_minus_control_pp'].items():
        seed_rows.append({'Modelo': report['model'], 'Controle': arm, 'Conjunto': report['dataset'],
                          'Sementes': len(effect['by_seed']), 'Ganho médio (pp)': round(effect['mean'], 2),
                          'Desvio-padrão (pp)': round(effect['sample_sd'], 2),
                          'Mínimo (pp)': round(effect['min'], 2), 'Máximo (pp)': round(effect['max'], 2)})
if seed_rows:
    table(seed_rows)
else:
    display(Markdown('Aguardando os resultados pareados das três sementes.'))
